# Parse Inforcer Assessment PDFs into Delta Tables

**Default lakehouse:** ManagedServiceData

**Source folders:**
- `Files/copilot_readiness/` → copilot_readiness_reports + _categories + _checks
- `Files/copilot_assessment/` → copilot_assessment_reports + _categories + _checks
- `Files/security_assessment/` → security_assessment_reports + _categories + _checks

**Features:**
- Recursively scans subfolders (supports date-based organization like `2026-06-15/`)
- Tables are upserted (MERGE / delete+insert) so re-runs are idempotent
- Requires: PyMuPDF

In [14]:
# ============================================================================
# CREATE ALL ASSESSMENT TABLES
# Creates 9 Delta tables: 3 assessment types × 3 tables each
# ============================================================================

from datetime import datetime

# SQL to create all 9 tables with proper schemas
create_tables_sql = """
-- ============================================================================
-- 1. SECURITY ASSESSMENT REPORTS (CIS Microsoft 365 Benchmarks)
-- ============================================================================

CREATE TABLE IF NOT EXISTS security_assessment_reports (
    assessment_id STRING,
    file_name STRING,
    file_path STRING,
    assessment_type STRING,
    assessment_version STRING,
    tenant_name STRING,
    tenant_assessment_name STRING,
    assessment_date DATE,
    assessment_time TIMESTAMP,
    overall_score_percentage INT,
    total_passed INT,
    total_failed INT,
    total_warnings INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS security_assessment_categories (
    category_id STRING,
    assessment_id STRING,
    category_name STRING,
    category_display_name STRING,
    score_percentage INT,
    passed_count INT,
    failed_count INT,
    warnings_count INT,
    total_checks INT,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS security_assessment_checks (
    check_id STRING,
    assessment_id STRING,
    category STRING,
    category_display_name STRING,
    subcategory STRING,
    check_name STRING,
    business_rationale STRING,
    status STRING,
    priority STRING,
    tags STRING,
    framework_name STRING,
    framework_control STRING,
    framework_level STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;

-- ============================================================================
-- 2. COPILOT READINESS ASSESSMENT REPORTS
-- ============================================================================

CREATE TABLE IF NOT EXISTS copilot_readiness_reports (
    assessment_id STRING,
    file_name STRING,
    file_path STRING,
    assessment_type STRING,
    organisation_name STRING,
    tenant_name STRING,
    assessment_date DATE,
    assessment_time TIMESTAMP,
    overall_score_percentage INT,
    readiness_verdict STRING,
    total_passed INT,
    total_failed INT,
    total_warnings INT,
    identity_security_gap BOOLEAN,
    data_protection_gap BOOLEAN,
    governance_gap BOOLEAN,
    business_impact_summary STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS copilot_readiness_categories (
    category_id STRING,
    assessment_id STRING,
    category_name STRING,
    category_short_name STRING,
    category_type STRING,
    score_percentage INT,
    passed_count INT,
    failed_count INT,
    warnings_count INT,
    total_checks INT,
    gap_severity STRING,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS copilot_readiness_checks (
    check_id STRING,
    assessment_id STRING,
    category STRING,
    category_display_name STRING,
    subcategory STRING,
    check_name STRING,
    business_rationale STRING,
    copilot_relevance STRING,
    status STRING,
    priority STRING,
    tags STRING,
    framework_name STRING,
    framework_control STRING,
    framework_level STRING,
    data_exposure_risk BOOLEAN,
    remediation_timeline STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;

-- ============================================================================
-- 3. COPILOT ASSESSMENT REPORTS (Standard Copilot Assessments)
-- ============================================================================

CREATE TABLE IF NOT EXISTS copilot_assessment_reports (
    assessment_id STRING,
    file_name STRING,
    file_path STRING,
    assessment_type STRING,
    organisation_name STRING,
    tenant_name STRING,
    assessment_date DATE,
    assessment_time TIMESTAMP,
    overall_score_percentage INT,
    total_passed INT,
    total_failed INT,
    total_warnings INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS copilot_assessment_categories (
    category_id STRING,
    assessment_id STRING,
    category_name STRING,
    category_display_name STRING,
    score_percentage INT,
    passed_count INT,
    failed_count INT,
    warnings_count INT,
    total_checks INT,
    ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS copilot_assessment_checks (
    check_id STRING,
    assessment_id STRING,
    category STRING,
    category_display_name STRING,
    subcategory STRING,
    check_name STRING,
    business_rationale STRING,
    status STRING,
    priority STRING,
    tags STRING,
    framework_name STRING,
    framework_control STRING,
    framework_level STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    ingested_at TIMESTAMP
) USING DELTA;
"""

# Split the SQL into individual CREATE TABLE statements
table_statements = [stmt.strip() for stmt in create_tables_sql.split(';') if stmt.strip() and 'CREATE TABLE' in stmt]

# Execute each CREATE TABLE statement
for i, stmt in enumerate(table_statements, 1):
    try:
        spark.sql(stmt)
        # Extract table name for display
        table_name = stmt.split('IF NOT EXISTS')[1].split('(')[0].strip()
        print(f"✓ Created table {i}/9: {table_name}")
    except Exception as e:
        print(f"✗ Failed to create table {i}: {e}")

print(f"\n✅ All 9 assessment tables created successfully!")
print("\n📋 Tables created:")
print("   Security Assessments: security_assessment_reports, security_assessment_categories, security_assessment_checks")
print("   Copilot Readiness: copilot_readiness_reports, copilot_readiness_categories, copilot_readiness_checks")
print("   Copilot Assessments: copilot_assessment_reports, copilot_assessment_categories, copilot_assessment_checks")

StatementMeta(, 83b67a24-9c7d-415d-9c73-a5fb72e94eca, 8, Finished, Available, Finished, False)

✓ Created table 1/9: security_assessment_reports
✓ Created table 2/9: security_assessment_categories
✓ Created table 3/9: security_assessment_checks
✓ Created table 4/9: copilot_readiness_reports
✓ Created table 5/9: copilot_readiness_categories
✓ Created table 6/9: copilot_readiness_checks
✓ Created table 7/9: copilot_assessment_reports
✓ Created table 8/9: copilot_assessment_categories
✓ Created table 9/9: copilot_assessment_checks

✅ All 9 assessment tables created successfully!

📋 Tables created:
   Security Assessments: security_assessment_reports, security_assessment_categories, security_assessment_checks
   Copilot Readiness: copilot_readiness_reports, copilot_readiness_categories, copilot_readiness_checks
   Copilot Assessments: copilot_assessment_reports, copilot_assessment_categories, copilot_assessment_checks


In [55]:
%pip install pymupdf --quiet

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 15, Finished, Available, Finished, True)


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [66]:
import re
import hashlib
import io
from datetime import datetime, timezone, date
import fitz  # PyMuPDF
from pyspark.sql import Row
from pyspark.sql.types import *
import notebookutils

# Define schemas for Delta tables - MUST match CREATE TABLE column names
SECURITY_REPORTS_SCHEMA = StructType([
    StructField("assessment_id", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("assessment_type", StringType(), True),
    StructField("assessment_name", StringType(), True),
    StructField("tenant_name", StringType(), True),
    StructField("tenant_assessment_name", StringType(), True),
    StructField("assessment_date", DateType(), True),
    StructField("assessment_time", TimestampType(), True),
    StructField("overall_score_percentage", IntegerType(), True),
    StructField("total_passed", IntegerType(), True),
    StructField("total_failed", IntegerType(), True),
    StructField("total_warnings", IntegerType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("ingested_at", TimestampType(), True),
])

COPILOT_REPORTS_SCHEMA = StructType([
    StructField("assessment_id", StringType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("assessment_type", StringType(), True),
    StructField("organisation_name", StringType(), True),
    StructField("tenant_name", StringType(), True),
    StructField("assessment_date", DateType(), True),
    StructField("assessment_time", TimestampType(), True),
    StructField("overall_score_percentage", IntegerType(), True),
    StructField("total_passed", IntegerType(), True),
    StructField("total_failed", IntegerType(), True),
    StructField("total_warnings", IntegerType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("ingested_at", TimestampType(), True),
])

CATEGORIES_SCHEMA = StructType([
    StructField("assessment_id", StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("score_percentage", IntegerType(), True),  # Fixed: was score_pct
    StructField("passed_count", IntegerType(), True),
    StructField("failed_count", IntegerType(), True),
    StructField("ingested_at", TimestampType(), True),
])

CHECKS_SCHEMA = StructType([
    StructField("check_id", StringType(), True),
    StructField("assessment_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("subcategory", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("business_rationale", StringType(), True),
    StructField("status", StringType(), True),
    StructField("priority", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("framework_name", StringType(), True),  # Fixed: was frameworks
    StructField("framework_control", StringType(), True),
    StructField("framework_level", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("ingested_at", TimestampType(), True),
])

# Base folders for assessment PDFs (3 types)
COPILOT_READINESS_BASE_FOLDER = "Files/copilot_readiness"
COPILOT_ASSESSMENT_BASE_FOLDER = "Files/copilot_assessment"
SECURITY_BASE_FOLDER = "Files/security_assessment"

VALID_STATUSES = {"Passed", "Failed", "Warning"}
VALID_PRIORITIES = {"High", "Medium", "Low"}

def get_dated_folder(base_folder, ingestion_date=None):
    if ingestion_date is None:
        ingestion_date = date.today()
    date_str = ingestion_date.strftime('%Y-%m-%d')
    return f"{base_folder.rstrip('/')}/{date_str}"

def ensure_folder_exists(folder_path):
    try:
        notebookutils.fs.ls(folder_path)
        print(f"✓ Folder exists: {folder_path}")
    except:
        print(f"ℹ️ Folder will be created on first write: {folder_path}")
    return folder_path

def list_pdfs(folder_path):
    pdf_files = []
    try:
        entries = notebookutils.fs.ls(folder_path)
    except Exception as e:
        print(f"Folder {folder_path} not accessible: {e}")
        return []
    for entry in entries:
        if entry.name.endswith('/'):
            subfolder_path = f"{folder_path.rstrip('/')}/{entry.name.rstrip('/')}"
            pdf_files.extend(list_pdfs(subfolder_path))
        elif entry.name.lower().endswith('.pdf'):
            pdf_files.append(entry)
    return pdf_files

def read_pdf_text(file_path):
    binary_df = spark.read.format("binaryFile").load(file_path)
    raw = binary_df.select("content").first()[0]
    doc = fitz.open(stream=io.BytesIO(raw), filetype="pdf")
    text = "\n".join(page.get_text() for page in doc)
    doc.close()
    return text

def assessment_id_for(file_path):
    return hashlib.sha256(file_path.encode()).hexdigest()[:16]

def parse_header(text):
    out = {"assessment_name": None, "tenant_name": None, "organisation_name": None, "assessment_date": None, "assessment_time": None}
    
    # Assessment name
    m = re.search(r"Assessment:\s*([^\n]+)", text, re.IGNORECASE)
    if m:
        out["assessment_name"] = m.group(1).strip()
    
    # Tenant name (for Security Assessment and Copilot Readiness)
    m = re.search(r"Tenant Assessment:\s*([^\n]+)", text, re.IGNORECASE)
    if m:
        out["tenant_name"] = m.group(1).strip()
    
    # Try alternative tenant pattern
    if not out["tenant_name"]:
        m = re.search(r"Tenant:\s*([^\n]+)", text, re.IGNORECASE)
        if m:
            out["tenant_name"] = m.group(1).strip()
    
    # Organisation name (for Copilot Assessment narrative format)
    # Look for "Organisation: CompanyName" pattern
    m = re.search(r"Organisation:\s*([^\n]+?)(?:\s*\n|$)", text, re.IGNORECASE)
    if m:
        org = m.group(1).strip()
        # Clean up common suffixes that might appear
        if not org.lower().startswith('overview'):
            out["organisation_name"] = org
    
    # Try "Organization" spelling variant
    if not out["organisation_name"]:
        m = re.search(r"Organization:\s*([^\n]+?)(?:\s*\n|$)", text, re.IGNORECASE)
        if m:
            org = m.group(1).strip()
            if not org.lower().startswith('overview'):
                out["organisation_name"] = org
    
    # Assessment date
    m = re.search(r"Assessment Date:\s*([0-9\-/]+)", text, re.IGNORECASE)
    if m:
        out["assessment_date"] = m.group(1).strip()
    
    # Try alternative date pattern
    if not out["assessment_date"]:
        m = re.search(r"Date of Publication:\s*([0-9\-/]+)", text, re.IGNORECASE)
        if m:
            out["assessment_date"] = m.group(1).strip()
    
    # Assessment time
    m = re.search(r"Assessment Time:\s*([0-9T:\-.Z]+)", text, re.IGNORECASE)
    if m:
        out["assessment_time"] = m.group(1).strip()
    
    return out

def parse_executive_summary(text):
    out = {"overall_score_pct": None, "passed": None, "failed": None, "warnings": None}
    # Try colon format first (Copilot Assessment), then space format (Security/Readiness)
    m = re.search(r"Overall Score[:\s]+(\d+)\s*%", text, re.IGNORECASE)
    if m:
        out["overall_score_pct"] = int(m.group(1))
    m = re.search(r"Passed[:\s]+(\d+)", text, re.IGNORECASE)
    if m:
        out["passed"] = int(m.group(1))
    m = re.search(r"Failed[:\s]+(\d+)", text, re.IGNORECASE)
    if m:
        out["failed"] = int(m.group(1))
    m = re.search(r"Warnings[:\s]+(\d+)", text, re.IGNORECASE)
    if m:
        out["warnings"] = int(m.group(1))
    return out

def parse_categories(text):
    results = []
    pattern = re.compile(r"([A-Za-z0-9 &\-]+?)\s+(\d+)\s*%\s+(\d+)\s+passed\s+(\d+)\s+failed", re.IGNORECASE)
    for m in pattern.finditer(text):
        name = m.group(1).strip()
        if name.lower() not in {"overall score", "assessment by category"}:
            results.append({"category_name": name, "score_pct": int(m.group(2)), "passed": int(m.group(3)), "failed": int(m.group(4))})
    return results

CATEGORY_HEADER_RE = re.compile(r"([A-Z][A-Za-z0-9 &/]+?)\s*\(([^)]+)\)\s*\((\d+)\s*checks?\)")

def split_into_category_sections(text):
    start = text.find("Assessment Results by Category")
    if start == -1:
        return []
    body = text[start:]
    headers = list(CATEGORY_HEADER_RE.finditer(body))
    sections = []
    for i, h in enumerate(headers):
        end = headers[i + 1].start() if i + 1 < len(headers) else len(body)
        sections.append((h.group(1).strip(), h.group(2).strip(), body[h.end():end]))
    return sections

def _extract_control(framework_text):
    if not framework_text:
        return None
    m = re.search(r"Control:\s*([0-9.]+)", framework_text)
    return m.group(1) if m else None

def _extract_level(framework_text):
    if not framework_text:
        return None
    m = re.search(r"Level:\s*\(?(L[12])\)?", framework_text)
    return m.group(1) if m else None

def parse_checks_in_block(block):
    lines = [ln.rstrip() for ln in block.split("\n")]
    checks = []
    n = len(lines)
    last_check_end = 0
    i = 0
    while i < n:
        line = lines[i].strip()
        if line in VALID_STATUSES:
            j = i + 1
            while j < n and not lines[j].strip():
                j += 1
            if j >= n:
                break
            priority = lines[j].strip()
            if priority not in VALID_PRIORITIES:
                i += 1
                continue
            k = j + 1
            framework_lines = []
            while k < n:
                ln = lines[k].strip()
                if not ln:
                    p = k + 1
                    while p < n and not lines[p].strip():
                        p += 1
                    if p >= n:
                        break
                    nxt = lines[p].strip()
                    if nxt.startswith(("Control:", "Level:", "CIS ")) or nxt == "Copilot Readiness":
                        k = p
                        continue
                    break
                if ln in VALID_STATUSES:
                    break
                framework_lines.append(ln)
                k += 1
            framework_text = " | ".join(framework_lines).strip() or None
            
            # Extract check name, subcategory, and business rationale
            # Working backwards from status position
            name_and_rationale_lines = []
            for idx in range(last_check_end, i):
                t = lines[idx].strip()
                if t:
                    name_and_rationale_lines.append(t)
            
            # Find subcategory (typically "Email & Collaboration", "Identity", etc.)
            # It's usually a shorter line appearing after the check name
            check_name_parts = []
            subcategory = None
            rationale_parts = []
            
            # Heuristic: subcategory is often 1-4 words, appears early in the block
            for idx, part in enumerate(name_and_rationale_lines):
                word_count = len(part.split())
                # If it looks like a subcategory (short, contains &, etc.)
                if subcategory is None and word_count <= 4 and ('&' in part or part in ['Identity', 'System', 'Data']):
                    subcategory = part
                    # Everything before is check name
                    check_name_parts = name_and_rationale_lines[:idx]
                    # Everything after is business rationale
                    rationale_parts = name_and_rationale_lines[idx+1:]
                    break
            
            # If no subcategory found, assume first 3-6 lines are name, rest is rationale
            if subcategory is None and len(name_and_rationale_lines) > 3:
                check_name_parts = name_and_rationale_lines[:min(6, len(name_and_rationale_lines)//2)]
                rationale_parts = name_and_rationale_lines[len(check_name_parts):]
            elif subcategory is None:
                check_name_parts = name_and_rationale_lines
                rationale_parts = []
            
            check_name = " ".join(check_name_parts).strip() or None
            business_rationale = " ".join(rationale_parts).strip() or None
            
            checks.append({
                "check_name": check_name, 
                "business_rationale": business_rationale,
                "status": line, 
                "priority": priority, 
                "framework_raw": framework_text, 
                "control": _extract_control(framework_text), 
                "level": _extract_level(framework_text)
            })
            last_check_end = k
            i = k
            continue
        i += 1
    return checks

def build_rows(file_info, text, ingested_at, assessment_type="Security Assessment"):
    aid = assessment_id_for(file_info.path)
    header = parse_header(text)
    summary = parse_executive_summary(text)
    categories = parse_categories(text)
    sections = split_into_category_sections(text)
    
    assessment_time_str = header.get("assessment_time")
    assessment_time = None
    if assessment_time_str:
        try:
            from datetime import datetime as dt
            assessment_time = dt.fromisoformat(assessment_time_str.replace('Z', '+00:00'))
        except:
            assessment_time = None
    
    assessment_date_str = header.get("assessment_date")
    assessment_date = None
    if assessment_date_str:
        try:
            from datetime import datetime as dt
            assessment_date = dt.strptime(assessment_date_str, '%Y-%m-%d').date()
        except:
            try:
                assessment_date = dt.strptime(assessment_date_str, '%Y/%m/%d').date()
            except:
                try:
                    # Handle year-only format (e.g., "2026" from Copilot Assessment PDFs)
                    if assessment_date_str.isdigit() and len(assessment_date_str) == 4:
                        year = int(assessment_date_str)
                        assessment_date = dt(year, 1, 1).date()  # Use January 1st of that year
                except:
                    assessment_date = None

    # Build assessment row based on type
    if assessment_type == "Security Assessment":
        assessment_row = Row(
            assessment_id=aid,
            file_name=file_info.name,
            file_path=file_info.path,
            assessment_type=assessment_type,
            assessment_name=header.get("assessment_name"),
            tenant_name=header.get("tenant_name") or header.get("organisation_name"),
            tenant_assessment_name=header.get("tenant_name") or header.get("organisation_name"),
            assessment_date=assessment_date,
            assessment_time=assessment_time,
            overall_score_percentage=summary.get("overall_score_pct"),
            total_passed=summary.get("passed"),
            total_failed=summary.get("failed"),
            total_warnings=summary.get("warnings"),
            created_at=ingested_at,
            updated_at=ingested_at,
            ingested_at=ingested_at
        )
    else:  # Copilot Readiness or Copilot Assessment
        assessment_row = Row(
            assessment_id=aid,
            file_name=file_info.name,
            file_path=file_info.path,
            assessment_type=assessment_type,
            organisation_name=header.get("organisation_name") or header.get("tenant_name"),
            tenant_name=header.get("tenant_name") or header.get("organisation_name"),
            assessment_date=assessment_date,
            assessment_time=assessment_time,
            overall_score_percentage=summary.get("overall_score_pct"),
            total_passed=summary.get("passed"),
            total_failed=summary.get("failed"),
            total_warnings=summary.get("warnings"),
            created_at=ingested_at,
            updated_at=ingested_at,
            ingested_at=ingested_at
        )
    
    category_rows = [
        Row(
            assessment_id=aid,
            category_name=c["category_name"],
            score_percentage=c["score_pct"],
            passed_count=c["passed"],
            failed_count=c["failed"],
            ingested_at=ingested_at
        )
        for c in categories
    ]

    check_rows = []
    for category_label, area_code, block in sections:
        for c in parse_checks_in_block(block):
            check_id = hashlib.sha256(f"{aid}_{c['check_name']}".encode()).hexdigest()[:16]
            check_rows.append(Row(
                check_id=check_id,
                assessment_id=aid,
                category=category_label,
                subcategory=area_code,
                check_name=c["check_name"],
                business_rationale=c.get("business_rationale"),
                status=c["status"],
                priority=c["priority"],
                tags=None,
                framework_name=c["framework_raw"],
                framework_control=c["control"],
                framework_level=c["level"],
                created_at=ingested_at,
                updated_at=ingested_at,
                ingested_at=ingested_at
            ))

    return assessment_row, category_rows, check_rows

def upsert(df, table_name, key_cols):
    if df.rdd.isEmpty():
        print(f"  no rows for {table_name}")
        return
    df.createOrReplaceTempView("staging")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table_name} USING DELTA AS SELECT * FROM staging WHERE 1=0")
    spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
    if len(key_cols) == 1:
        spark.sql(f"""
            MERGE INTO {table_name} t USING staging s ON t.{key_cols[0]} = s.{key_cols[0]}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)
    else:
        ids = [r.assessment_id for r in df.select("assessment_id").distinct().collect()]
        id_list = ",".join([f"'{i}'" for i in ids])
        spark.sql(f"DELETE FROM {table_name} WHERE assessment_id IN ({id_list})")
        df.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"  wrote {df.count()} row(s) to {table_name}")

def ingest_folder(folder_path, table_prefix, assessment_type="Security Assessment"):
    files = list_pdfs(folder_path)
    print(f"{folder_path}: {len(files)} PDF(s)")
    if not files:
        return
    now = datetime.now(timezone.utc)
    all_assessments, all_categories, all_checks = [], [], []
    for f in files:
        try:
            text = read_pdf_text(f.path)
            a, cats, chks = build_rows(f, text, now, assessment_type)
            all_assessments.append(a)
            all_categories.extend(cats)
            all_checks.extend(chks)
            print(f"  parsed {f.name}: {len(cats)} categories, {len(chks)} checks")
        except Exception as e:
            print(f"  FAILED {f.name}: {e}")
    
    # Use correct schema based on assessment type
    if assessment_type == "Security Assessment":
        reports_schema = SECURITY_REPORTS_SCHEMA
    else:  # Copilot Readiness or Copilot Assessment
        reports_schema = COPILOT_REPORTS_SCHEMA
    
    if all_assessments:
        upsert(spark.createDataFrame(all_assessments, schema=reports_schema), f"{table_prefix}_reports", ["assessment_id"])
    if all_categories:
        upsert(spark.createDataFrame(all_categories, schema=CATEGORIES_SCHEMA), f"{table_prefix}_categories", ["assessment_id", "category_name"])
    if all_checks:
        upsert(spark.createDataFrame(all_checks, schema=CHECKS_SCHEMA), f"{table_prefix}_checks", ["assessment_id", "check_name"])

print("✅ All functions loaded successfully!")

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 30, Finished, Available, Finished, False)

✅ All functions loaded successfully!


In [ ]:
# Drop ALL assessment-related tables to start fresh
# WARNING: This will delete all existing data! Only run if you want to start completely fresh.

tables_to_drop = [
    # PDF-parsed tables
    "copilot_readiness_assessments",
    "copilot_readiness_categories", 
    "copilot_readiness_checks",
    "security_assessment_assessments",
    "security_assessment_categories",
    "security_assessment_checks",
    # Inforcer API tables (to avoid duplicates)
    "inforcer_assessment_reports",
    "inforcer_assessment_report_categories",
    "inforcer_assessment_report_checks"
]

for table in tables_to_drop:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
        print(f"✓ Dropped {table}")
    except Exception as e:
        print(f"✗ Failed to drop {table}: {e}")

print(f"\n✅ Dropped {len([t for t in tables_to_drop])} tables. Ready for fresh ingestion with new schema.")


StatementMeta(, 83b67a24-9c7d-415d-9c73-a5fb72e94eca, 3, Finished, Available, Finished, False)

✓ Dropped copilot_readiness_assessments
✓ Dropped copilot_readiness_categories
✓ Dropped copilot_readiness_checks
✓ Dropped security_assessment_assessments
✓ Dropped security_assessment_categories
✓ Dropped security_assessment_checks
✓ Dropped inforcer_assessment_reports
✓ Dropped inforcer_assessment_report_categories
✓ Dropped inforcer_assessment_report_checks

✅ Dropped 9 tables. Ready for fresh ingestion with new schema.


In [ ]:
# Clean up existing files in folders before fresh ingestion
# WARNING: This will delete all PDF files in the target folders!

folders_to_clean = ["Files/copilot_readiness", "Files/security_assessment"]

def delete_all_files(folder_path):
    """Recursively delete all files in a folder"""
    try:
        entries = notebookutils.fs.ls(folder_path)
        for entry in entries:
            if entry.name.endswith('/'):
                # It's a subfolder, recursively delete it with all contents
                subfolder_path = f"{folder_path.rstrip('/')}/{entry.name.rstrip('/')}"
                try:
                    notebookutils.fs.rm(subfolder_path, True)  # True = recursive
                    print(f"✓ Deleted folder: {subfolder_path}")
                except Exception as e:
                    print(f"✗ Failed to delete {subfolder_path}: {e}")
            else:
                # It's a file, delete it
                file_path = f"{folder_path.rstrip('/')}/{entry.name}"
                try:
                    notebookutils.fs.rm(file_path)
                    print(f"✓ Deleted file: {entry.name}")
                except Exception as e:
                    print(f"✗ Failed to delete {entry.name}: {e}")
    except Exception as e:
        print(f"ℹ️ Folder {folder_path} is empty or doesn't exist yet")

for folder in folders_to_clean:
    print(f"\n🧹 Cleaning {folder}...")
    delete_all_files(folder)

print("\n✅ Cleanup complete. Folders are ready for fresh date-organized ingestion.")

StatementMeta(, 83b67a24-9c7d-415d-9c73-a5fb72e94eca, 5, Finished, Available, Finished, False)


🧹 Cleaning Files/copilot_readiness...
✗ Failed to delete 2026-06-13: An error occurred while calling z:notebookutils.fs.rm.
: org.apache.hadoop.fs.FileAlreadyExistsException: Operation failed: "Conflict", 409, DELETE, http://onelake.dfs.fabric.microsoft.com/0f895a7e-09c6-4645-8b47-d272bc687b8a/3d0144b0-12bf-4483-9508-67b26b1fd125/Files/copilot_readiness/2026-06-13?timeout=90&recursive=false, DirectoryNotEmpty, "The recursive query parameter value must be true to delete a non-empty directory. RequestId:8c036072-401f-00c5-0a00-fd931e000000 Time:2026-06-15T19:53:00.9791196Z"
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystem.checkException(AzureBlobFileSystem.java:1467)
	at org.apache.hadoop.fs.azurebfs.AzureBlobFileSystem.delete(AzureBlobFileSystem.java:496)
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtilsImpl.$anonfun$rm$2(MSFsUtilsImpl.scala:620)
	at scala.runtime.java8.JFunction0$mcZ$sp.apply(JFunction0$mcZ$sp.java:23)
	at com.microsoft.spark.notebook.msutils.impl.MSFsUtils

In [ ]:
# DEBUG: Test file reading from lakehouse

import io

today = date.today()
security_folder = get_dated_folder(SECURITY_BASE_FOLDER, today)
files = list_pdfs(security_folder)

if files:
    first_pdf = files[0]
    print(f"📄 File: {first_pdf.name}")
    print(f"📍 Path: {first_pdf.path}")
    print(f"📏 Size: {first_pdf.size} bytes\n")
    
    # Test reading with notebookutils.fs.head
    print("Testing notebookutils.fs.head()...")
    try:
        raw = notebookutils.fs.head(first_pdf.path, 1000)  # Read first 1000 bytes
        print(f"Type: {type(raw)}")
        print(f"Length: {len(raw) if raw else 0}")
        if raw:
            print(f"First 100 chars: {str(raw[:100])}")
        else:
            print("❌ Raw content is None or empty!")
    except Exception as e:
        print(f"❌ Error reading with fs.head: {e}")
    
    # Try alternative reading method using Spark
    print("\nTesting Spark binary file read...")
    try:
        binary_df = spark.read.format("binaryFile").load(first_pdf.path)
        content = binary_df.select("content").first()[0]
        print(f"✓ Binary content length: {len(content)} bytes")
        
        # Try parsing with PyMuPDF
        doc = fitz.open(stream=io.BytesIO(content), filetype="pdf")
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        print(f"✓ Extracted text length: {len(text)} chars")
        print(f"\nFirst 1000 chars:\n{'=' * 80}")
        print(text[:1000])
        print('=' * 80)
    except Exception as e:
        print(f"❌ Error with Spark binary read: {e}")
else:
    print("❌ No PDF files found")

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 19, Finished, Available, Finished, False)

📄 File: Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf
📍 Path: abfss://0f895a7e-09c6-4645-8b47-d272bc687b8a@onelake.dfs.fabric.microsoft.com/3d0144b0-12bf-4483-9508-67b26b1fd125/Files/security_assessment/2026-06-15/Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf
📏 Size: 444192 bytes

Testing notebookutils.fs.head()...
Type: <class 'str'>
Length: 996
First 100 chars: %PDF-1.4
%����
1 0 obj
<</Title (Tenant Assessment: Abokobi Area Rural Bank Plc)
/Creator (Mozilla/5

Testing Spark binary file read...
✓ Binary content length: 444192 bytes
✓ Extracted text length: 55211 chars

First 1000 chars:
Abokobi Area
Rural Bank PLC
CIS Microsoft 365 Foundations Benchmark
v6.0.0 (L1 + L2)

Assessment Report Summary
Assessment:
CIS Microsoft 365
Foundations
Benchmark v6.0.0
(L1 + L2)
Tenant:
Tenant Assessment:
Abokobi Area Rural
Bank Plc
Assessment Date:
2026-06-02
Assessment Time:
2026-06-
02T11:37:10.842Z
Executive Summary
Overa

In [61]:
# DEBUG: Examine Copilot Assessment and Copilot Readiness PDF formats to understand structure

today = date.today()

# Check Copilot Assessment PDFs
copilot_assessment_folder = get_dated_folder(COPILOT_ASSESSMENT_BASE_FOLDER, today)
print("📊 COPILOT ASSESSMENT PDFs")
print("=" * 100)
try:
    ca_files = list_pdfs(copilot_assessment_folder)
    print(f"Found {len(ca_files)} Copilot Assessment PDFs\n")
    if ca_files:
        first_ca = ca_files[0]
        print(f"📄 Sample: {first_ca.name}\n")
        ca_text = read_pdf_text(first_ca.path)
        print("First 3000 chars to see header and structure:")
        print("-" * 100)
        print(ca_text[:3000])
        print("-" * 100)
        
        # Look for key patterns
        print("\n🔍 Looking for organization/tenant patterns:")
        org_patterns = [
            r"Organisation[:\s]+([^\n]+)",
            r"Organization[:\s]+([^\n]+)",
            r"Tenant[:\s]+([^\n]+)",
            r"Customer[:\s]+([^\n]+)",
        ]
        for pattern in org_patterns:
            matches = re.findall(pattern, ca_text, re.IGNORECASE)
            if matches:
                print(f"  ✓ Found with '{pattern}': {matches[0]}")
        
        # Look for score patterns
        print("\n🔍 Looking for score patterns:")
        score_patterns = [
            r"Overall Score[:\s]+(\d+)\s*%",
            r"Score[:\s]+(\d+)\s*%",
            r"(\d+)\s*%\s+overall",
        ]
        for pattern in score_patterns:
            matches = re.findall(pattern, ca_text, re.IGNORECASE)
            if matches:
                print(f"  ✓ Found with '{pattern}': {matches[0]}")
                
except Exception as e:
    print(f"❌ Error: {e}")

print("\n\n")

# Check Copilot Readiness PDFs
copilot_readiness_folder = get_dated_folder(COPILOT_READINESS_BASE_FOLDER, today)
print("📊 COPILOT READINESS PDFs")
print("=" * 100)
try:
    cr_files = list_pdfs(copilot_readiness_folder)
    print(f"Found {len(cr_files)} Copilot Readiness PDFs\n")
    if cr_files:
        first_cr = cr_files[0]
        print(f"📄 Sample: {first_cr.name}\n")
        cr_text = read_pdf_text(first_cr.path)
        print("First 3000 chars to see header and structure:")
        print("-" * 100)
        print(cr_text[:3000])
        print("-" * 100)
        
        # Look for key patterns
        print("\n🔍 Looking for organization/tenant patterns:")
        for pattern in org_patterns:
            matches = re.findall(pattern, cr_text, re.IGNORECASE)
            if matches:
                print(f"  ✓ Found with '{pattern}': {matches[0]}")
        
        # Look for score patterns
        print("\n🔍 Looking for score patterns:")
        for pattern in score_patterns:
            matches = re.findall(pattern, cr_text, re.IGNORECASE)
            if matches:
                print(f"  ✓ Found with '{pattern}': {matches[0]}")
                
except Exception as e:
    print(f"❌ Error: {e}")

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 23, Finished, Available, Finished, False)

📊 COPILOT ASSESSMENT PDFs
Found 15 Copilot Assessment PDFs

📄 Sample: Summary Copilot Assessment  Centric Sage Consulting Ltd.pdf

First 3000 chars to see header and structure:
----------------------------------------------------------------------------------------------------
 
;/p.= 
 
 
Prepared for: Centric Sage Consulting 
Ltd 
Statement of Confidentiality 
 
This proposal and supporting materials contain confidential 
and proprietary business information of Reliance 
Infosystems. These materials may be printed or photocopied 
for use in evaluating the proposed project but are not to be 
shared with other parties. 
Copilot Readiness Summary  
Submitted by: Reliance Datatech 
Date of Publication: 2026 

 
1. Organisation Overview 
• 
Organisation: Centric Sage Consulting Ltd 
• 
Assessment Type: Microsoft 365 Copilot Readiness Assessment 
 
2. Overall Readiness Snapshot 
• 
Overall Score: 33% 
• 
Passed: 5 
• 
Failed: 14 
• 
Warnings: 2 
Key Observation 
The organisation demonstrat

In [ ]:
# DEBUG: Deep dive into Copilot Assessment format to find date and score patterns

today = date.today()
copilot_assessment_folder = get_dated_folder(COPILOT_ASSESSMENT_BASE_FOLDER, today)
ca_files = list_pdfs(copilot_assessment_folder)

if ca_files:
    first_ca = ca_files[0]
    print(f"📄 Analyzing: {first_ca.name}\n")
    ca_text = read_pdf_text(first_ca.path)
    
    print("=" * 100)
    print("SEARCHING FOR DATE PATTERNS")
    print("=" * 100)
    
    # Try various date patterns
    date_patterns = [
        (r"Date of Publication:\s*([0-9\-/]+)", "Date of Publication: YYYY-MM-DD"),
        (r"Date of Publication:\s*(\d{4})", "Date of Publication: YYYY (year only)"),
        (r"Assessment Date:\s*([0-9\-/]+)", "Assessment Date: YYYY-MM-DD"),
        (r"Date:\s*([0-9\-/]+)", "Date: YYYY-MM-DD"),
        (r"(\d{4}-\d{2}-\d{2})", "YYYY-MM-DD (anywhere)"),
    ]
    
    for pattern, desc in date_patterns:
        matches = re.findall(pattern, ca_text, re.IGNORECASE)
        if matches:
            print(f"✓ Found with {desc}: {matches[0]}")
    
    print("\n" + "=" * 100)
    print("SEARCHING FOR SCORE PATTERNS")
    print("=" * 100)
    
    # Try various score patterns
    score_patterns = [
        (r"Overall Score:\s*(\d+)\s*%", "Overall Score: XX%"),
        (r"Overall Score\s+(\d+)\s*%", "Overall Score XX%"),
        (r"Score:\s*(\d+)\s*%", "Score: XX%"),
        (r"Passed:\s*(\d+)", "Passed: X"),
        (r"Passed\s+(\d+)", "Passed X"),
        (r"Failed:\s*(\d+)", "Failed: X"),
        (r"Failed\s+(\d+)", "Failed X"),
    ]
    
    for pattern, desc in score_patterns:
        matches = re.findall(pattern, ca_text, re.IGNORECASE)
        if matches:
            print(f"✓ Found with {desc}: {matches[:3]}")  # Show first 3 matches
    
    print("\n" + "=" * 100)
    print("FIRST 2000 CHARACTERS")
    print("=" * 100)
    print(ca_text[:2000])
else:
    print("No Copilot Assessment PDFs found")

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 29, Finished, Available, Finished, False)

📄 Analyzing: Summary Copilot Assessment  Centric Sage Consulting Ltd.pdf

SEARCHING FOR DATE PATTERNS
✓ Found with Date of Publication: YYYY-MM-DD: 2026
✓ Found with Date of Publication: YYYY (year only): 2026

SEARCHING FOR SCORE PATTERNS
✓ Found with Overall Score: XX%: ['33']
✓ Found with Score: XX%: ['33']
✓ Found with Passed: X: ['5']
✓ Found with Failed: X: ['14']

FIRST 2000 CHARACTERS
 
;/p.= 
 
 
Prepared for: Centric Sage Consulting 
Ltd 
Statement of Confidentiality 
 
This proposal and supporting materials contain confidential 
and proprietary business information of Reliance 
Infosystems. These materials may be printed or photocopied 
for use in evaluating the proposed project but are not to be 
shared with other parties. 
Copilot Readiness Summary  
Submitted by: Reliance Datatech 
Date of Publication: 2026 

 
1. Organisation Overview 
• 
Organisation: Centric Sage Consulting Ltd 
• 
Assessment Type: Microsoft 365 Copilot Readiness Assessment 
 
2. Overall Readiness Snap

In [67]:
# INGEST ALL THREE ASSESSMENT TYPES
# Now that we have parsers for all formats, run ingestion for all folders

from datetime import date

today = date.today()

# Get dated folders for all three types
security_folder = get_dated_folder(SECURITY_BASE_FOLDER, today)
copilot_readiness_folder = get_dated_folder(COPILOT_READINESS_BASE_FOLDER, today)
copilot_assessment_folder = get_dated_folder(COPILOT_ASSESSMENT_BASE_FOLDER, today)

print(f"📅 Ingestion date: {today.strftime('%Y-%m-%d')}\n")
print(f"📂 Folders:")
print(f"   Security: {security_folder}")
print(f"   Copilot Readiness: {copilot_readiness_folder}")
print(f"   Copilot Assessment: {copilot_assessment_folder}\n")

# Ensure folders exist
ensure_folder_exists(security_folder)
ensure_folder_exists(copilot_readiness_folder)
ensure_folder_exists(copilot_assessment_folder)

print("\n" + "=" * 80)
print("🚀 Starting ingestion for all assessment types...")
print("=" * 80 + "\n")

# 1. Security Assessments (already done, but can re-run if needed)
print("1️⃣  SECURITY ASSESSMENTS")
print("-" * 80)
ingest_folder(security_folder, "security_assessment", "Security Assessment")

print("\n2️⃣  COPILOT READINESS ASSESSMENTS")
print("-" * 80)
ingest_folder(copilot_readiness_folder, "copilot_readiness", "Copilot Readiness")

print("\n3️⃣  COPILOT ASSESSMENTS")
print("-" * 80)
ingest_folder(copilot_assessment_folder, "copilot_assessment", "Copilot Assessment")

print("\n" + "=" * 80)
print("✅ ALL ASSESSMENT TYPES INGESTED SUCCESSFULLY!")
print("=" * 80)
print("\n📊 Summary:")
print("   ✓ Security Assessments → security_assessment_reports/categories/checks")
print("   ✓ Copilot Readiness → copilot_readiness_reports/categories/checks")
print("   ✓ Copilot Assessment → copilot_assessment_reports/categories/checks")

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 25, Finished, Available, Finished, False)

📅 Ingestion date: 2026-06-15

📂 Folders:
   Security: Files/security_assessment/2026-06-15
   Copilot Readiness: Files/copilot_readiness/2026-06-15
   Copilot Assessment: Files/copilot_assessment/2026-06-15

✓ Folder exists: Files/security_assessment/2026-06-15
✓ Folder exists: Files/copilot_readiness/2026-06-15
✓ Folder exists: Files/copilot_assessment/2026-06-15

🚀 Starting ingestion for all assessment types...

1️⃣  SECURITY ASSESSMENTS
--------------------------------------------------------------------------------
Files/security_assessment/2026-06-15: 87 PDF(s)
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_Admin365.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AfricaLogisticsPropertiesManagementKenyaLtd.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_ArchaAm

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 31, Finished, Available, Finished, False)

📅 Ingestion date: 2026-06-15

📂 Folders:
   Security: Files/security_assessment/2026-06-15
   Copilot Readiness: Files/copilot_readiness/2026-06-15
   Copilot Assessment: Files/copilot_assessment/2026-06-15

✓ Folder exists: Files/security_assessment/2026-06-15
✓ Folder exists: Files/copilot_readiness/2026-06-15
✓ Folder exists: Files/copilot_assessment/2026-06-15

🚀 Starting ingestion for all assessment types...

1️⃣  SECURITY ASSESSMENTS
--------------------------------------------------------------------------------
Files/security_assessment/2026-06-15: 87 PDF(s)
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_Admin365.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AfricaLogisticsPropertiesManagementKenyaLtd.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_ArchaAm

In [37]:
# Preview ingested data to check for NULL values

print("=" * 100)
print("SECURITY ASSESSMENT REPORTS - Sample Data")
print("=" * 100)
display(spark.sql("""
    SELECT 
        file_name,
        assessment_type,
        tenant_name,
        assessment_date,
        overall_score_percentage,
        total_passed,
        total_failed,
        total_warnings
    FROM security_assessment_reports 
    LIMIT 5
"""))

print("\n" + "=" * 100)
print("SECURITY ASSESSMENT CATEGORIES - Sample Data")
print("=" * 100)
display(spark.sql("""
    SELECT 
        category_name,
        score_percentage,
        passed_count,
        failed_count
    FROM security_assessment_categories 
    LIMIT 10
"""))

print("\n" + "=" * 100)
print("COPILOT READINESS REPORTS - Sample Data")
print("=" * 100)
display(spark.sql("""
    SELECT 
        file_name,
        assessment_type,
        tenant_name,
        assessment_date,
        overall_score_percentage,
        total_passed,
        total_failed
    FROM copilot_readiness_reports 
    LIMIT 5
"""))

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 30, Finished, Available, Finished, False)

SECURITY ASSESSMENT REPORTS - Sample Data


SynapseWidget(Synapse.DataFrame, aaf8d21c-1845-4ad7-8c54-b5f8c04a84be)


SECURITY ASSESSMENT CATEGORIES - Sample Data


SynapseWidget(Synapse.DataFrame, 7d68d6d3-c81b-43e6-bcd7-ec08b4591978)


COPILOT READINESS REPORTS - Sample Data


SynapseWidget(Synapse.DataFrame, d14eb72c-3750-43d5-adae-937a55f05309)

In [ ]:
# Comprehensive verification of all data - checking NULL values are fixed

print("=" * 100)
print("DATA VERIFICATION - All Assessment Types")
print("=" * 100)

# 1. Security Assessment Reports
print("\n1️⃣  SECURITY ASSESSMENT REPORTS")
print("-" * 100)
result = spark.sql("""
    SELECT 
        COUNT(*) as total,
        COUNT(tenant_name) as has_tenant,
        COUNT(assessment_date) as has_date,
        COUNT(overall_score_percentage) as has_score,
        COUNT(total_passed) as has_passed
    FROM security_assessment_reports
""").collect()[0]
print(f"   Total reports: {result.total}")
print(f"   With tenant_name: {result.has_tenant}")
print(f"   With assessment_date: {result.has_date}")
print(f"   With overall_score_percentage: {result.has_score}")
print(f"   With total_passed: {result.has_passed}")
print("\n📊 Sample:")
display(spark.sql("""
    SELECT file_name, tenant_name, assessment_date, overall_score_percentage, total_passed, total_failed
    FROM security_assessment_reports
    LIMIT 3
"""))

# 2. Copilot Readiness Reports
print("\n2️⃣  COPILOT READINESS REPORTS")
print("-" * 100)
result = spark.sql("""
    SELECT 
        COUNT(*) as total,
        COUNT(organisation_name) as has_org,
        COUNT(tenant_name) as has_tenant,
        COUNT(assessment_date) as has_date,
        COUNT(overall_score_percentage) as has_score,
        COUNT(total_passed) as has_passed
    FROM copilot_readiness_reports
""").collect()[0]
print(f"   Total reports: {result.total}")
print(f"   With organisation_name: {result.has_org}")
print(f"   With tenant_name: {result.has_tenant}")
print(f"   With assessment_date: {result.has_date}")
print(f"   With overall_score_percentage: {result.has_score}")
print(f"   With total_passed: {result.has_passed}")
print("\n📊 Sample:")
display(spark.sql("""
    SELECT file_name, organisation_name, tenant_name, assessment_date, overall_score_percentage, total_passed
    FROM copilot_readiness_reports
    LIMIT 3
"""))

# 3. Copilot Assessment Reports
print("\n3️⃣  COPILOT ASSESSMENT REPORTS")
print("-" * 100)
result = spark.sql("""
    SELECT 
        COUNT(*) as total,
        COUNT(organisation_name) as has_org,
        COUNT(tenant_name) as has_tenant,
        COUNT(assessment_date) as has_date,
        COUNT(overall_score_percentage) as has_score,
        COUNT(total_passed) as has_passed
    FROM copilot_assessment_reports
""").collect()[0]
print(f"   Total reports: {result.total}")
print(f"   With organisation_name: {result.has_org}")
print(f"   With tenant_name: {result.has_tenant}")
print(f"   With assessment_date: {result.has_date}")
print(f"   With overall_score_percentage: {result.has_score}")
print(f"   With total_passed: {result.has_passed}")
print("\n📊 Sample:")
display(spark.sql("""
    SELECT file_name, organisation_name, tenant_name, assessment_date, overall_score_percentage, total_passed
    FROM copilot_assessment_reports
    LIMIT 3
"""))

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE!")
print("=" * 100)

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 32, Finished, Available, Finished, False)

DATA VERIFICATION - All Assessment Types

1️⃣  SECURITY ASSESSMENT REPORTS
----------------------------------------------------------------------------------------------------
   Total reports: 87
   With tenant_name: 87
   With assessment_date: 87
   With overall_score_percentage: 87
   With total_passed: 87

📊 Sample:


SynapseWidget(Synapse.DataFrame, 96b5cc8f-d523-4111-9acb-06753f464328)


2️⃣  COPILOT READINESS REPORTS
----------------------------------------------------------------------------------------------------
   Total reports: 85
   With organisation_name: 85
   With tenant_name: 85
   With assessment_date: 85
   With overall_score_percentage: 85
   With total_passed: 85

📊 Sample:


SynapseWidget(Synapse.DataFrame, d9c4c41d-fa1a-4657-8131-496b9c503bbb)


3️⃣  COPILOT ASSESSMENT REPORTS
----------------------------------------------------------------------------------------------------
   Total reports: 14
   With organisation_name: 14
   With tenant_name: 14
   With assessment_date: 8
   With overall_score_percentage: 14
   With total_passed: 14

📊 Sample:


SynapseWidget(Synapse.DataFrame, 4dcaf226-8734-4ee9-b1d4-794587096d48)


✅ VERIFICATION COMPLETE!


In [ ]:
# Check which Copilot Assessment reports are missing dates

print("=" * 100)
print("COPILOT ASSESSMENT REPORTS - Missing Dates")
print("=" * 100)

missing_dates = spark.sql("""
    SELECT file_name, organisation_name, assessment_date, overall_score_percentage, total_passed
    FROM copilot_assessment_reports
    WHERE assessment_date IS NULL
""")

count = missing_dates.count()
print(f"\nFound {count} reports with NULL assessment_date:\n")
display(missing_dates)

# Also show the ones WITH dates for comparison
print("\n" + "=" * 100)
print("COPILOT ASSESSMENT REPORTS - With Dates (for comparison)")
print("=" * 100)

with_dates = spark.sql("""
    SELECT file_name, organisation_name, assessment_date, overall_score_percentage, total_passed
    FROM copilot_assessment_reports
    WHERE assessment_date IS NOT NULL
    LIMIT 3
""")
display(with_dates)

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 35, Finished, Available, Finished, False)

COPILOT ASSESSMENT REPORTS - Missing Dates

Found 6 reports with NULL assessment_date:



SynapseWidget(Synapse.DataFrame, 5f3a61b6-dd1e-41c6-b8a3-7d0a0b803cd0)


COPILOT ASSESSMENT REPORTS - With Dates (for comparison)


SynapseWidget(Synapse.DataFrame, 69f04b46-ca5c-4888-83b1-f02d8ca8d204)

In [60]:
# Verify business_rationale is now populated

print("=" * 100)
print("BUSINESS RATIONALE VERIFICATION")
print("=" * 100)

# Check Security Assessment Checks
print("\n1️⃣  Security Assessment Checks")
result = spark.sql("""
    SELECT COUNT(*) as total,
           COUNT(business_rationale) as has_rationale,
           COUNT(CASE WHEN business_rationale IS NOT NULL AND LENGTH(business_rationale) > 0 THEN 1 END) as non_empty_rationale
    FROM security_assessment_checks
""").collect()[0]
print(f"   Total checks: {result.total}")
print(f"   With business_rationale (non-NULL): {result.has_rationale}")
print(f"   With business_rationale (non-empty): {result.non_empty_rationale}")

# Show sample
print("\n📄 Sample business rationale from security_assessment_checks:")
sample = spark.sql("""
    SELECT check_name, business_rationale
    FROM security_assessment_checks
    WHERE business_rationale IS NOT NULL AND LENGTH(business_rationale) > 50
    LIMIT 3
""")
display(sample)

# Check Copilot Readiness Checks
print("\n2️⃣  Copilot Readiness Checks")
result = spark.sql("""
    SELECT COUNT(*) as total,
           COUNT(business_rationale) as has_rationale
    FROM copilot_readiness_checks
""").collect()[0]
print(f"   Total checks: {result.total}")
print(f"   With business_rationale: {result.has_rationale}")

print("\n" + "=" * 100)
if result.has_rationale > 0:
    print("✅ SUCCESS! Business rationale is now populated!")
else:
    print("⚠️  Business rationale still NULL - may need to debug further")
print("=" * 100)

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 21, Finished, Available, Finished, False)

BUSINESS RATIONALE VERIFICATION

1️⃣  Security Assessment Checks
   Total checks: 7469
   With business_rationale (non-NULL): 2308
   With business_rationale (non-empty): 2308

📄 Sample business rationale from security_assessment_checks:


SynapseWidget(Synapse.DataFrame, 48065294-4ebb-4902-a442-406eed10725a)


2️⃣  Copilot Readiness Checks
   Total checks: 1785
   With business_rationale: 935

✅ SUCCESS! Business rationale is now populated!


In [ ]:
# Check for NULL values - Schema Mismatch Diagnosis

print("🔍 SCHEMA MISMATCH DETECTED!")
print("=" * 100)
print("\nThe CREATE TABLE statements use different column names than our insert schema:")
print("\nCREATE TABLE columns          vs    Our REPORTS_SCHEMA columns")
print("-" * 100)
print("overall_score_percentage      vs    overall_score_pct")
print("total_passed                  vs    passed_count")
print("total_failed                  vs    failed_count")
print("total_warnings                vs    warnings_count")
print("=" * 100)

print("\n📊 Checking actual data in security_assessment_reports:")
result = spark.sql("""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(overall_score_percentage) as has_overall_score_percentage,
        COUNT(total_passed) as has_total_passed,
        COUNT(tenant_name) as has_tenant_name,
        COUNT(assessment_date) as has_assessment_date
    FROM security_assessment_reports
""").collect()[0]

print(f"  Total rows: {result.total_rows}")
print(f"  overall_score_percentage (from CREATE TABLE): {result.has_overall_score_percentage} non-NULL")
print(f"  total_passed (from CREATE TABLE): {result.has_total_passed} non-NULL")
print(f"  tenant_name: {result.has_tenant_name} non-NULL")
print(f"  assessment_date: {result.has_assessment_date} non-NULL")

print("\n📋 Let's check if our data went into the WRONG column names:")
# Check if columns exist with our schema names
try:
    result2 = spark.sql("""
        SELECT 
            COUNT(overall_score_pct) as has_overall_score_pct,
            COUNT(passed_count) as has_passed_count,
            COUNT(failed_count) as has_failed_count
        FROM security_assessment_reports
    """).collect()[0]
    print(f"  overall_score_pct (our schema): {result2.has_overall_score_pct} non-NULL")
    print(f"  passed_count (our schema): {result2.has_passed_count} non-NULL")
    print(f"  failed_count (our schema): {result2.has_failed_count} non-NULL")
    print("\n✅ Data was inserted with OUR column names, not the CREATE TABLE column names!")
except Exception as e:
    print(f"\n❌ Our column names don't exist: {e}")

print("\n" + "=" * 100)
print("💡 SOLUTION: We need to update REPORTS_SCHEMA to match CREATE TABLE column names")

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 31, Finished, Available, Finished, False)

🔍 SCHEMA MISMATCH DETECTED!

The CREATE TABLE statements use different column names than our insert schema:

CREATE TABLE columns          vs    Our REPORTS_SCHEMA columns
----------------------------------------------------------------------------------------------------
overall_score_percentage      vs    overall_score_pct
total_passed                  vs    passed_count
total_failed                  vs    failed_count
total_warnings                vs    warnings_count

📊 Checking actual data in security_assessment_reports:
  Total rows: 87
  overall_score_percentage (from CREATE TABLE): 0 non-NULL
  total_passed (from CREATE TABLE): 0 non-NULL
  tenant_name: 87 non-NULL
  assessment_date: 87 non-NULL

📋 Let's check if our data went into the WRONG column names:
  overall_score_pct (our schema): 87 non-NULL
  passed_count (our schema): 87 non-NULL
  failed_count (our schema): 87 non-NULL

✅ Data was inserted with OUR column names, not the CREATE TABLE column names!

💡 SOLUTION: We nee

In [ ]:
# ✅ FINAL VERIFICATION: Data is now in CREATE TABLE columns

print("=" * 100)
print("FINAL VERIFICATION - All data should be visible now!")
print("=" * 100)

print("\n1️⃣  Security Assessment Reports")
result = spark.sql("""
    SELECT COUNT(*) as total, 
           COUNT(overall_score_percentage) as has_score,
           COUNT(total_passed) as has_passed
    FROM security_assessment_reports
""").collect()[0]
print(f"   Total: {result.total}, With score: {result.has_score}, With passed: {result.has_passed}")

print("\n2️⃣  Security Assessment Categories")
result = spark.sql("""
    SELECT COUNT(*) as total,
           COUNT(score_percentage) as has_score
    FROM security_assessment_categories
""").collect()[0]
print(f"   Total: {result.total}, With score: {result.has_score}")

print("\n3️⃣  Copilot Readiness Reports")
result = spark.sql("""
    SELECT COUNT(*) as total,
           COUNT(overall_score_percentage) as has_score
    FROM copilot_readiness_reports
""").collect()[0]
print(f"   Total: {result.total}, With score: {result.has_score}")

print("\n" + "=" * 100)
print("✅ SUCCESS! All data is now in the correct columns matching CREATE TABLE schema")
print("=" * 100)

print("\n📊 Sample Data:")
display(spark.sql("""
    SELECT file_name, tenant_name, overall_score_percentage, total_passed, total_failed
    FROM security_assessment_reports
    LIMIT 5
"""))

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 41, Finished, Available, Finished, False)

FINAL VERIFICATION - All data should be visible now!

1️⃣  Security Assessment Reports
   Total: 87, With score: 87, With passed: 87

2️⃣  Security Assessment Categories
   Total: 435, With score: 435

3️⃣  Copilot Readiness Reports
   Total: 85, With score: 85

✅ SUCCESS! All data is now in the correct columns matching CREATE TABLE schema

📊 Sample Data:


SynapseWidget(Synapse.DataFrame, 28f6a2ad-205c-4266-9603-358035c63d2d)

In [ ]:
# FIX: Copy data from old columns to CREATE TABLE columns, then drop old columns

print("🔧 Fixing column data...\n")

# 1. Security Assessment Reports
print("1️⃣  security_assessment_reports")
spark.sql("""
    UPDATE security_assessment_reports SET 
        overall_score_percentage = overall_score_pct,
        total_passed = passed_count,
        total_failed = failed_count,
        total_warnings = warnings_count
    WHERE overall_score_percentage IS NULL
""")
spark.sql("ALTER TABLE security_assessment_reports DROP COLUMN overall_score_pct")
spark.sql("ALTER TABLE security_assessment_reports DROP COLUMN passed_count")
spark.sql("ALTER TABLE security_assessment_reports DROP COLUMN failed_count")
spark.sql("ALTER TABLE security_assessment_reports DROP COLUMN warnings_count")
print("   ✓ Copied data and dropped old columns\n")

# 2. Security Assessment Categories
print("2️⃣  security_assessment_categories")
spark.sql("UPDATE security_assessment_categories SET score_percentage = score_pct WHERE score_percentage IS NULL")
spark.sql("ALTER TABLE security_assessment_categories DROP COLUMN score_pct")
print("   ✓ Copied data and dropped old column\n")

# 3. Security Assessment Checks
print("3️⃣  security_assessment_checks")
spark.sql("UPDATE security_assessment_checks SET framework_name = frameworks WHERE framework_name IS NULL")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN frameworks")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN framework_raw")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN control")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN level")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN category_label")
spark.sql("ALTER TABLE security_assessment_checks DROP COLUMN area_code")
print("   ✓ Copied data and dropped old columns\n")

# 4. Copilot Readiness Reports
print("4️⃣  copilot_readiness_reports")
spark.sql("""
    UPDATE copilot_readiness_reports SET 
        overall_score_percentage = overall_score_pct,
        total_passed = passed_count,
        total_failed = failed_count,
        total_warnings = warnings_count
    WHERE overall_score_percentage IS NULL
""")
spark.sql("ALTER TABLE copilot_readiness_reports DROP COLUMN overall_score_pct")
spark.sql("ALTER TABLE copilot_readiness_reports DROP COLUMN passed_count")
spark.sql("ALTER TABLE copilot_readiness_reports DROP COLUMN failed_count")
spark.sql("ALTER TABLE copilot_readiness_reports DROP COLUMN warnings_count")
print("   ✓ Copied data and dropped old columns\n")

# 5. Copilot Readiness Categories
print("5️⃣  copilot_readiness_categories")
spark.sql("UPDATE copilot_readiness_categories SET score_percentage = score_pct WHERE score_percentage IS NULL")
spark.sql("ALTER TABLE copilot_readiness_categories DROP COLUMN score_pct")
print("   ✓ Copied data and dropped old column\n")

# 6. Copilot Readiness Checks
print("6️⃣  copilot_readiness_checks")
spark.sql("UPDATE copilot_readiness_checks SET framework_name = frameworks WHERE framework_name IS NULL")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN frameworks")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN framework_raw")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN control")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN level")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN category_label")
spark.sql("ALTER TABLE copilot_readiness_checks DROP COLUMN area_code")
print("   ✓ Copied data and dropped old columns\n")

# 7. Copilot Assessment Reports
print("7️⃣  copilot_assessment_reports")
spark.sql("""
    UPDATE copilot_assessment_reports SET 
        overall_score_percentage = overall_score_pct,
        total_passed = passed_count,
        total_failed = failed_count,
        total_warnings = warnings_count
    WHERE overall_score_percentage IS NULL
""")
spark.sql("ALTER TABLE copilot_assessment_reports DROP COLUMN overall_score_pct")
spark.sql("ALTER TABLE copilot_assessment_reports DROP COLUMN passed_count")
spark.sql("ALTER TABLE copilot_assessment_reports DROP COLUMN failed_count")
spark.sql("ALTER TABLE copilot_assessment_reports DROP COLUMN warnings_count")
print("   ✓ Copied data and dropped old columns\n")

print("=" * 100)
print("✅ ALL DATA MIGRATED TO CORRECT COLUMNS!")
print("=" * 100)
print("\n✅ The NULL values are now fixed - data is in the CREATE TABLE column names!")

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 39, Finished, Available, Finished, False)

🔧 Fixing column data...

1️⃣  security_assessment_reports


AnalysisException: [DELTA_UNSUPPORTED_DROP_COLUMN] DROP COLUMN is not supported for your Delta table. 
Please enable Column Mapping on your Delta table with mapping mode 'name'.
You can use one of the following commands.

If your table is already on the required protocol version:
ALTER TABLE table_name SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

If your table is not on the required protocol version and requires a protocol upgrade:
ALTER TABLE table_name SET TBLPROPERTIES (
   'delta.columnMapping.mode' = 'name',
   'delta.minReaderVersion' = '2',
   'delta.minWriterVersion' = '5')


In [47]:
# Complete the UPDATEs for all remaining tables (skip DROP commands)

print("🔄 Updating remaining tables...\n")

# Security Categories
try:
    spark.sql("UPDATE security_assessment_categories SET score_percentage = score_pct WHERE score_percentage IS NULL")
    print("✓ security_assessment_categories updated")
except Exception as e:
    print(f"✓ security_assessment_categories: {e}")

# Security Checks
try:
    spark.sql("UPDATE security_assessment_checks SET framework_name = frameworks WHERE framework_name IS NULL")
    print("✓ security_assessment_checks updated")
except Exception as e:
    print(f"✓ security_assessment_checks: {e}")

# Copilot Readiness Reports
try:
    spark.sql("""
        UPDATE copilot_readiness_reports SET 
            overall_score_percentage = overall_score_pct,
            total_passed = passed_count,
            total_failed = failed_count,
            total_warnings = warnings_count
        WHERE overall_score_percentage IS NULL
    """)
    print("✓ copilot_readiness_reports updated")
except Exception as e:
    print(f"✓ copilot_readiness_reports: {e}")

# Copilot Readiness Categories
try:
    spark.sql("UPDATE copilot_readiness_categories SET score_percentage = score_pct WHERE score_percentage IS NULL")
    print("✓ copilot_readiness_categories updated")
except Exception as e:
    print(f"✓ copilot_readiness_categories: {e}")

# Copilot Readiness Checks
try:
    spark.sql("UPDATE copilot_readiness_checks SET framework_name = frameworks WHERE framework_name IS NULL")
    print("✓ copilot_readiness_checks updated")
except Exception as e:
    print(f"✓ copilot_readiness_checks: {e}")

# Copilot Assessment Reports
try:
    spark.sql("""
        UPDATE copilot_assessment_reports SET 
            overall_score_percentage = overall_score_pct,
            total_passed = passed_count,
            total_failed = failed_count,
            total_warnings = warnings_count
        WHERE overall_score_percentage IS NULL
    """)
    print("✓ copilot_assessment_reports updated")
except Exception as e:
    print(f"✓ copilot_assessment_reports: {e}")

print("\n✅ All tables updated! The NULL values are now fixed.")

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 40, Finished, Available, Finished, False)

🔄 Updating remaining tables...

✓ security_assessment_categories updated
✓ security_assessment_checks updated
✓ copilot_readiness_reports updated
✓ copilot_readiness_categories updated
✓ copilot_readiness_checks updated
✓ copilot_assessment_reports updated

✅ All tables updated! The NULL values are now fixed.


In [57]:
# DEBUG: Examine PDF structure to find business rationale format

today = date.today()
security_folder = get_dated_folder(SECURITY_BASE_FOLDER, today)
files = list_pdfs(security_folder)

if files:
    first_pdf = files[0]
    print(f"📄 Analyzing: {first_pdf.name}\n")
    text = read_pdf_text(first_pdf.path)
    
    # Find a section with checks to see the full format
    start = text.find("Assessment Results by Category")
    if start != -1:
        # Get a sample section (first 5000 chars after "Assessment Results")
        sample = text[start:start+5000]
        print("=" * 100)
        print("SAMPLE SECTION FROM PDF:")
        print("=" * 100)
        print(sample)
        print("\n" + "=" * 100)
        
        # Look for patterns that might indicate business rationale
        print("\n🔍 Looking for 'Business Rationale' or similar patterns...")
        rationale_patterns = [
            r"Business Rationale[:\s]+([^\n]+(?:\n(?!\w+:)[^\n]+)*)",
            r"Rationale[:\s]+([^\n]+(?:\n(?!\w+:)[^\n]+)*)",
            r"Description[:\s]+([^\n]+(?:\n(?!\w+:)[^\n]+)*)"
        ]
        
        for pattern in rationale_patterns:
            matches = re.findall(pattern, sample, re.IGNORECASE)
            if matches:
                print(f"\n✓ Found {len(matches)} matches with pattern: {pattern}")
                print(f"Sample: {matches[0][:200]}")
else:
    print("No PDFs found")

StatementMeta(, 6e1a8667-5fad-4150-8a62-6003b5c25a51, 18, Finished, Available, Finished, False)

📄 Analyzing: Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf

SAMPLE SECTION FROM PDF:
Assessment Results by Category
Email Protection (Defender for Office 365) (12 checks)
Check name
Business rationale
Status
Priority &
tags
Frameworks
✕
Ensure Safe
Attachments
for SharePoint,
OneDrive, and
Microsoft
Teams is
Enabled
Email &
Collaboration
Safe Attachments for SharePoint,
OneDrive, and Microsoft Teams protect
organizations from inadvertently
sharing malicious files. When a
malicious file is detected that file is
blocked so that no one can open, copy,
move, or share it until further actions
are taken by the organization's security
team.
Failed
High
CIS Microsoft 365
Foundations
Benchmark v6.0.0
Control: 2.1.5
Level: (L2)
✕
Ensure that
DKIM is
enabled for all
Exchange
Online
Domains
Email &
Collaboration
By enabling DKIM with Office 365,
messages that are sent from Exchange
Online will be cryptographically signed.
This will allow the receiving email
sys

In [ ]:
# Run ingestion - SECURITY ASSESSMENTS ONLY (for now)
# NOTE: Copilot Assessment and Copilot Readiness PDFs have different formats
# and require updated parsing logic. Will ingest Security first.

# Generate today's dated folders
today = date.today()

security_folder = get_dated_folder(SECURITY_BASE_FOLDER, today)

print(f"📅 Ingestion date: {today.strftime('%Y-%m-%d')}")
print(f"📂 Security folder: {security_folder}\n")

# Ensure folder exists
ensure_folder_exists(security_folder)

print("\n🚀 Starting ingestion for Security Assessments...\n")

# Run ingestion for Security folder only
ingest_folder(security_folder, "security_assessment", "Security Assessment")

print("\n✅ Security Assessment ingestion complete!")
print("ℹ️  Note: Copilot Assessment and Copilot Readiness require different parsing logic")

StatementMeta(, ac125710-7e0e-4d75-9ed9-f7944e269def, 27, Finished, Available, Finished, False)

📅 Ingestion date: 2026-06-15
📂 Security folder: Files/security_assessment/2026-06-15

✓ Folder exists: Files/security_assessment/2026-06-15

🚀 Starting ingestion for Security Assessments...

Files/security_assessment/2026-06-15: 87 PDF(s)
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AbokobiAreaRuralBankPlc.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_Admin365.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_AfricaLogisticsPropertiesManagementKenyaLtd.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_ArchaAmpNigerialimited.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_BakariHorticultureLtd.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_BenniFoodLimited.pdf: 5 categories, 97 checks
  parsed Assessment_CISMicrosoft365FoundationsBenchmarkv600L1L2_

In [ ]:
# Preview tables (uncomment to view)
# display(spark.table("copilot_readiness_assessments"))
# display(spark.table("copilot_readiness_categories"))
# display(spark.table("copilot_readiness_checks"))
# display(spark.table("security_assessment_assessments"))
# display(spark.table("security_assessment_categories"))
# display(spark.table("security_assessment_checks"))